# 03. Station context (Version 2)

What is around each station? This notebook counts places from OpenStreetMap within a walking radius of every station, and measures the land-use mix, so the map can explain *why* a station is busy on weekdays or at weekends.

| Section | What it does |
|---|---|
| 0 | Settings: radius, place types |
| 1 | Load stations and draw a circle around each |
| 2 | Fetch OpenStreetMap features (once, then cached) |
| 3 | Count place types per station |
| 4 | Land-use mix per station |
| 5 | Check: do the counts match the weekend ratio? |
| 6 | Save the outputs |

Needs the outputs of `01datapreparation.ipynb` in `data/processed`. The first run downloads OpenStreetMap data and can take several minutes; later runs read the saved copy in seconds.

**Note on the data:** OpenStreetMap is mapped by volunteers. Counts show what has been *mapped*, not an official census. Central Kuala Lumpur is well covered; some outer areas may be undercounted.

## 0. Settings

In [1]:
from pathlib import Path
import json
from datetime import datetime
import pandas as pd
import geopandas as gpd
import osmnx as ox

RADIUS_M = 500            # <- walking radius around each station, in metres
REFRESH_OSM = False       # <- True = fetch OpenStreetMap again (e.g. a few months later)

# Place types to count. Each entry: OpenStreetMap key -> list of values (True = any value).
# To add places of worship later, add:  "worship": {"amenity": ["place_of_worship"]},
PLACE_TYPES = {
    "shops":      {"shop": True},
    "food":       {"amenity": ["restaurant", "cafe", "fast_food", "food_court"]},
    "offices":    {"office": True},
    "education":  {"amenity": ["school", "college", "university"]},
    "healthcare": {"amenity": ["hospital", "clinic", "doctors"]},
    "tourism_leisure": {"tourism": ["attraction", "museum", "gallery", "hotel"], "leisure": ["park"]},
}

# Land-use groups. Each entry: OpenStreetMap key -> list of values.
LAND_USE_GROUPS = {
    "residential": {"landuse": ["residential"]},
    "commercial":  {"landuse": ["commercial", "retail"]},
    "industrial":  {"landuse": ["industrial"]},
    "green":       {"landuse": ["grass", "forest", "recreation_ground", "meadow", "village_green"],
                    "leisure": ["park"]},
}

PROCESSED = Path("data/processed")
RAW = Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)
OSM_CACHE = RAW / f"osm_context_{RADIUS_M}m.parquet"

METRIC_CRS = "EPSG:32647"   # UTM zone 47N: measures in metres, suits the Klang Valley
print(f"Radius {RADIUS_M} m | {len(PLACE_TYPES)} place types | {len(LAND_USE_GROUPS)} land-use groups")

Radius 500 m | 6 place types | 4 land-use groups


## 1. Stations and their circles

In [2]:
stations = gpd.read_file(PROCESSED / "stations.geojson")
stations_m = stations.to_crs(METRIC_CRS)
circles = stations_m[["key", "name", "geometry"]].copy()
circles["geometry"] = stations_m.geometry.buffer(RADIUS_M)
circle_area = 3.14159265 * RADIUS_M ** 2

# One shape covering all circles, used to fetch OpenStreetMap data only where needed
search_area = circles.to_crs("EPSG:4326").union_all()
print(f"{len(circles)} stations; search area covers {circles.union_all().area / 1e6:.0f} km2 "
      f"(circles overlap where stations are close together)")

154 stations; search area covers 110 km2 (circles overlap where stations are close together)


## 2. Fetch OpenStreetMap features

One request fetches every tag we need. The result is reduced to a few columns and saved in `data/raw`, so this step is skipped on later runs unless `REFRESH_OSM = True`.

In [3]:
def all_tags():
    """Merge the tags of every place type and land-use group into one request."""
    tags = {}
    for group in list(PLACE_TYPES.values()) + list(LAND_USE_GROUPS.values()):
        for key, values in group.items():
            if values is True or tags.get(key) is True:
                tags[key] = True
            else:
                tags[key] = sorted(set(tags.get(key, [])) | set(values))
    return tags

KEEP = ["shop", "amenity", "office", "tourism", "leisure", "landuse", "name"]

if REFRESH_OSM or not OSM_CACHE.exists():
    print("Fetching from OpenStreetMap (this can take several minutes)...")
    osm = ox.features_from_polygon(search_area, tags=all_tags())
    osm = osm[[c for c in KEEP if c in osm.columns] + ["geometry"]].reset_index()
    osm = osm[osm.geometry.geom_type.isin(["Point", "Polygon", "MultiPolygon"])]
    for c in KEEP:
        if c not in osm.columns:
            osm[c] = None
        osm[c] = osm[c].astype("string")
    osm["fetched_on"] = datetime.now().strftime("%Y-%m-%d")
    osm.to_parquet(OSM_CACHE)
else:
    print(f"Using saved copy {OSM_CACHE}")
osm = gpd.read_parquet(OSM_CACHE)
print(f"{len(osm):,} features, fetched on {osm['fetched_on'].iloc[0]}")
print(osm.geom_type.value_counts().to_string())

Fetching from OpenStreetMap (this can take several minutes)...
12,463 features, fetched on 2026-09-30
Point           9047
Polygon         3359
MultiPolygon      57


## 3. Count place types

A place counts for a station if its centre point lies inside that station's circle. A place can count for two stations when their circles overlap, which is correct: it is within walking distance of both.

In [4]:
def matches(df, rules):
    """True for rows matching any key/value rule, e.g. {"amenity": ["cafe"]}."""
    hit = pd.Series(False, index=df.index)
    for key, values in rules.items():
        col = df[key]
        hit |= col.notna() if values is True else col.isin(values)
    return hit

points = osm.to_crs(METRIC_CRS)
points["geometry"] = points.geometry.representative_point()   # a mall polygon becomes one point

counts = pd.DataFrame(index=circles["key"])
for place, rules in PLACE_TYPES.items():
    pts = points[matches(points, rules)]
    joined = gpd.sjoin(pts[["geometry"]], circles[["key", "geometry"]], predicate="within")
    counts[place] = joined.groupby("key").size()
    print(f"{place:16} {len(pts):6,} mapped in the search area")
counts = counts.fillna(0).astype(int)

counts["total_places"] = counts.sum(axis=1)
display(counts.sort_values("total_places", ascending=False).head(8))

shops             4,452 mapped in the search area
food              3,373 mapped in the search area
offices             915 mapped in the search area
education           318 mapped in the search area
healthcare          374 mapped in the search area
tourism_leisure     707 mapped in the search area


,shops,food,offices,education,healthcare,tourism_leisure,total_places
key,,,,,,,
chowkit,458,148,50,7,23,54,740
bukitbintang,283,337,19,1,7,88,735
masjidjamek,364,167,33,8,10,57,639
imbi,237,263,18,1,7,83,609
pasarseni,254,195,31,4,3,51,538
pwtc,327,125,35,5,12,29,533
rajachulan,138,228,19,0,3,42,430
medantuanku,142,155,41,10,5,23,376


## 4. Land-use mix

For each station, the share of its circle covered by each land-use group. Groups can overlap slightly (a park inside a residential area), and part of a circle may have no land use mapped at all, so the shares need not add up to 100%.

In [5]:
land = osm[osm.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs(METRIC_CRS)
land["geometry"] = land.geometry.buffer(0)   # repairs occasional invalid shapes

shares = pd.DataFrame(index=circles["key"])
for group, rules in LAND_USE_GROUPS.items():
    polys = land[matches(land, rules)]
    if polys.empty:
        shares[group] = 0.0
        continue
    merged = gpd.GeoDataFrame(geometry=[polys.union_all()], crs=METRIC_CRS)
    inter = gpd.overlay(circles[["key", "geometry"]], merged, how="intersection")
    shares[group] = inter.set_index("key").area / circle_area
shares = shares.fillna(0).clip(upper=1).round(3)
shares.columns = [f"share_{c}" for c in shares.columns]

any_group = pd.Series(False, index=land.index)
for rules in LAND_USE_GROUPS.values():
    any_group |= matches(land, rules)
mapped_any = gpd.GeoDataFrame(geometry=[land[any_group].union_all()], crs=METRIC_CRS)
shares["share_mapped"] = (gpd.overlay(circles[["key", "geometry"]], mapped_any, how="intersection")
                          .set_index("key").area / circle_area).reindex(shares.index).fillna(0).clip(upper=1).round(3)

# Main land use: the largest group, if it covers at least 10% of the circle
share_cols = [f"share_{g}" for g in LAND_USE_GROUPS]
main = shares[share_cols].idxmax(axis=1).str.replace("share_", "")
shares["main_land_use"] = main.where(shares[share_cols].max(axis=1) >= 0.10, "not mapped")

print(shares["main_land_use"].value_counts().to_string())
print(f"Median share of each circle with land use mapped: {shares['share_mapped'].median():.0%}")

main_land_use
residential    85
commercial     46
industrial     11
not mapped      9
green           3
Median share of each circle with land use mapped: 47%


## 5. Check: do the counts match the weekend ratio?

If the weekend-ratio colours mean what we think, commuter stations (low ratio) should have more offices and education nearby, and weekend stations (high ratio) more shops, food and leisure. A rank correlation near zero means no link; negative means more of that place type goes with a lower (more commuter-like) ratio.

In [6]:
context = stations.set_index("key")[["name", "weekend_ratio", "avg_daily_entries"]].join(counts).join(shares)

check_cols = list(PLACE_TYPES) + share_cols
# Rank correlation (Spearman) computed with pandas only: correlate the ranks
corr = context[check_cols].rank().corrwith(context["weekend_ratio"].rank()).round(2)
print("Rank correlation with weekend ratio (negative = more commuter-like):")
print(corr.sort_values().to_string())

print("\nMost commuter-like stations and their surroundings:")
display(context.nsmallest(5, "weekend_ratio")[["name", "weekend_ratio"] + list(PLACE_TYPES) + ["main_land_use"]])
print("Most weekend-heavy stations and their surroundings:")
display(context.nlargest(5, "weekend_ratio")[["name", "weekend_ratio"] + list(PLACE_TYPES) + ["main_land_use"]])

Rank correlation with weekend ratio (negative = more commuter-like):
share_residential   -0.18
share_green         -0.06
share_industrial    -0.04
offices             -0.00
education            0.03
tourism_leisure      0.04
food                 0.11
shops                0.15
share_commercial     0.16
healthcare           0.17

Most commuter-like stations and their surroundings:


,name,weekend_ratio,shops,food,offices,education,healthcare,tourism_leisure,main_land_use
key,,,,,,,,,
kerinchi,Kerinchi,0.266,17,34,98,5,3,9,commercial
semantan,Semantan,0.323,7,17,4,0,1,1,residential
rajachulan,Raja Chulan,0.336,138,228,19,0,3,42,commercial
sultanismail,Sultan Ismail,0.344,135,79,26,9,5,22,commercial
asiajaya,Asia Jaya,0.366,7,17,9,5,1,3,residential


Most weekend-heavy stations and their surroundings:


,name,weekend_ratio,shops,food,offices,education,healthcare,tourism_leisure,main_land_use
key,,,,,,,,,
pasarseni,Pasar Seni,1.270,254,195,31,4,3,51,commercial
jambatankota,Jambatan Kota,1.200,26,17,4,0,1,5,commercial
bandarbukittinggi,Bandar Bukit Tinggi,1.187,7,8,4,1,3,0,residential
imbi,Imbi,1.168,237,263,18,1,7,83,commercial
johansetia,Johan Setia,1.102,3,2,0,0,0,1,residential


## 6. Save the outputs

In [7]:
out = counts.join(shares).reset_index()
out.to_csv(PROCESSED / "station_context.csv", index=False)

# Land-use areas inside the circles, for an optional map layer (simplified to keep the page light)
lu_rows = []
for group, rules in LAND_USE_GROUPS.items():
    polys = land[matches(land, rules)][["geometry"]].copy()
    polys["land_use"] = group
    lu_rows.append(polys)
landuse = gpd.GeoDataFrame(pd.concat(lu_rows, ignore_index=True), crs=METRIC_CRS)
landuse = gpd.clip(landuse, circles.union_all())
landuse = landuse.dissolve(by="land_use").reset_index()
landuse["geometry"] = landuse.geometry.simplify(5)
landuse.to_crs("EPSG:4326").to_file(PROCESSED / "landuse_near_stations.geojson", driver="GeoJSON")

meta = {"radius_m": RADIUS_M, "osm_fetched_on": str(osm["fetched_on"].iloc[0]),
        "place_types": PLACE_TYPES, "land_use_groups": LAND_USE_GROUPS,
        "prepared_on": datetime.now().strftime("%Y-%m-%d"),
        "source": "OpenStreetMap contributors (ODbL)",
        "note": "Counts reflect what volunteers have mapped in OpenStreetMap, not an official census."}
(PROCESSED / "context_metadata.json").write_text(json.dumps(meta, indent=2))

for f in ["station_context.csv", "landuse_near_stations.geojson", "context_metadata.json"]:
    print(f"{f:32} {(PROCESSED / f).stat().st_size / 1e3:8.0f} KB")

station_context.csv                    11 KB
landuse_near_stations.geojson         557 KB
context_metadata.json                   1 KB


___
**END**